# Week 1: Python & NumPy Mastery
## Notebook 10: NumPy Universal Functions (ufuncs) & Vectorize Internals

### 1. Conceptual Intuition & Theoretical Background
NumPy's execution speed stems from **Universal Functions (ufuncs)**: compiled C-functions that operate element-by-element across n-dimensional arrays.

A ufunc consists of:
1. **Loop Dispatch Table:** Maps input dtypes (e.g., `float32`, `int64`) to optimized C loops.
2. **Inner 1D Loop:** Executes contiguous pointer arithmetic without Python object checking:
$$\text{Output}[i] = f(\text{Input}_1[i \cdot s_1],\ \text{Input}_2[i \cdot s_2])$$
3. **Broadcasting Engine:** Automatically stretches singleton dimensions without memory duplication.

While `np.vectorize` provides a convenient Python wrapper, it is merely a syntactic sugar over a Python loop. In contrast, genuine C-level ufuncs (or `np.frompyfunc`) implement true elementwise vector dispatch with minimal overhead.

---

### 2. Scratch Python Implementation: Custom Universal Function Engine

In [ ]:
import numpy as np
import time

# 1. Custom Elementwise Activation Function: Leaky Swish
# f(x) = x * sigmoid(x) if x >= 0 else 0.01 * x
def scalar_leaky_swish(x):
    sig = 1.0 / (1.0 + np.exp(-max(-30.0, min(30.0, float(x)))))
    return x * sig if x >= 0 else 0.01 * x

# Create true ufunc via np.frompyfunc (takes 1 input, returns 1 output)
ufunc_leaky_swish = np.frompyfunc(scalar_leaky_swish, 1, 1)

# Pure vectorized NumPy implementation using native ufuncs
def native_numpy_swish(arr):
    sig = 1.0 / (1.0 + np.exp(-np.clip(arr, -30.0, 30.0)))
    return np.where(arr >= 0, arr * sig, 0.01 * arr)

# Benchmark on large array
data = np.random.uniform(-5.0, 5.0, size=50000)

t0 = time.perf_counter()
res_native = native_numpy_swish(data)
t_native = time.perf_counter() - t0

t0 = time.perf_counter()
res_ufunc = ufunc_leaky_swish(data).astype(np.float64)
t_ufunc = time.perf_counter() - t0

# Validate mathematical equivalence
max_diff = np.max(np.abs(res_native - res_ufunc))

print("--- NumPy ufunc Benchmark ---")
print(f"Native Vectorized Runtime: {t_native * 1000:.2f} ms")
print(f"Frompyfunc Dispatch Time:  {t_ufunc * 1000:.2f} ms")
print(f"Max Absolute Discrepancy:  {max_diff:.1e}")
print(f"Native vectorization is {t_ufunc / t_native:.1f}x faster due to compiled SIMD loops!")

### 3. Step-by-Step Code Walkthrough
1. **Loop Dispatch:** `np.frompyfunc` wraps a scalar Python callable into a generalized broadcasting ufunc interface.
2. **Native SIMD Compilation:** `native_numpy_swish` composes built-in C-level ufuncs (`np.exp`, `np.where`), executing compiled vectorized instructions.
3. **Precision Clamping:** `np.clip(..., -30, 30)` guarantees numerical stability against floating point overflow.

---

### 4. Industry Applications & Trade-offs
- **Custom Loss Functions & Activations:** Defining specialized neural network activations in edge inference runtimes without PyTorch overhead.
- **Trade-off:** `np.frompyfunc` still invokes the Python interpreter for each element; for maximum throughput, use Cython or Numba.

---

### 5. Top Interview Questions & Answers
**Q1: Does `np.vectorize` compile Python functions to machine code?**
> **Answer:** No. `np.vectorize` is primarily for convenience. It executes a standard Python `for` loop internally and does not accelerate execution.

**Q2: What is the difference between a ufunc and a standard Python function in NumPy?**
> **Answer:** A ufunc provides built-in broadcasting, type coercion, and dimensional reduction methods such as `.reduce()`, `.accumulate()`, and `.outer()` implemented directly in C.